# 00 — ChEMBL Target ID ile ligand ve regression veri store oluşturma

Bu notebook bir **ChEMBL Target ID** alır, hedefe ait `IC50` aktivitelerini ChEMBL REST API üzerinden indirir, uygun **single protein format** assay kayıtlarını seçer, kesin aktivite ölçümlerini tutar, ligand SMILES yapılarını dikkatli biçimde doğrular ve parent yapı bazında temiz bir regression veri seti oluşturur.

Ana çıktı:

```text
{TARGET_ID}_IC50_single_protein_format_CLEAN_parent_dedup.csv
```

Bu CSV Google Drive'a kaydedildikten sonra GitHub deposundaki `data/` klasörüne yüklenir.

## Workshop veri akışı

Bu seri aşağıdaki akışı kullanır:

1. Notebook çalıştırılır.
2. Üretilen dosyalar Google Drive içindeki aynı workshop klasörüne kaydedilir.
3. İlgili ana çıktı GitHub deposunun `data/` klasörüne yüklenir.
4. Sonraki notebook girdiyi GitHub RAW bağlantısından okur.

GitHub deposu:

```text
https://github.com/MOL-FEST/MOL-FET_regression/tree/main
```

Google Drive çıktı klasörü:

```text
MyDrive/MOL_FET_regression_workshop/
```

## Bilimsel dönüşüm

Ham IC50 değeri molar birime çevrildikten sonra:

\[
pIC_{50}=-\log_{10}(IC_{50}[M])
\]

ChEMBL kaydında geçerli `pchembl_value` varsa öncelikle bu değer kullanılır. Yoksa `standard_value` ve `standard_units` alanlarından pStandard hesaplanır.

## Küçük hücreli workshop sürümü

Bu sürümde çalışan kodun bilimsel ve teknik mantığı değiştirilmemiştir.  
Yalnızca uzun ana kod hücresi, workshop sırasında adım adım çalıştırılabilmesi ve hatanın kolay bulunabilmesi için daha küçük hücrelere ayrılmıştır.

> Hücreleri yukarıdan aşağıya sırayla çalıştırınız.

## Hücre 1 — Paket kontrolü

Temel paketler kontrol edilir. Eski `rdkit-pypi` yerine güncel `rdkit` paketi kullanılır.

In [ ]:
import sys
# Notebook içinden pip komutunu aktif Python yorumlayıcısıyla çalıştırmak için kullanılır.

import subprocess
# Eksik paketleri Colab ortamına kurmak için kullanılır.

import importlib.util
# Bir Python modülünün kurulu olup olmadığını kontrol etmek için kullanılır.

packages = [
    ("numpy", "numpy"),
    # Sayısal işlemler ve pStandard dönüşümü için kullanılır.

    ("pandas", "pandas"),
    # ChEMBL kayıtlarını tablo halinde düzenlemek ve CSV kaydetmek için kullanılır.

    ("requests", "requests"),
    # ChEMBL REST API isteklerini göndermek için kullanılır.

    ("tqdm", "tqdm"),
    # Uzun assay ve molekül işlemlerinde ilerleme çubuğu göstermek için kullanılır.
]
# Temel paketler import adı ve pip adı çiftleri halinde tanımlanır.

for import_name, pip_name in packages:
    # Her paket sırayla kontrol edilir.

    if importlib.util.find_spec(import_name) is None:
        # Paket bulunamazsa kurulum adımına geçilir.

        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        )
        # Paket aktif Colab Python ortamına kurulur.

if importlib.util.find_spec("rdkit") is None:
    # RDKit mevcut değilse güncel PyPI paket adı kontrol edilir.

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "rdkit"]
    )
    # Eski ve Python 3.12'de hata verebilen rdkit-pypi yerine güncel rdkit paketi kurulur.

print("✅ CHECKPOINT 00.1: Paket kontrolü tamamlandı; rdkit-pypi kullanılmadı.")
# Paket hücresinin başarıyla tamamlandığı bildirilir.

## Hücre 2 — Kütüphanelerin içe aktarılması

Dosya yolları, ChEMBL API, tablo işlemleri, ilerleme çubuğu, Google Drive ve RDKit araçları içe aktarılır.

In [ ]:
from pathlib import Path
# Google Drive ve çıktı dosyalarının yollarını güvenli biçimde yönetmek için kullanılır.

import time
# Ardışık ChEMBL API çağrıları arasında kısa bekleme vermek için kullanılır.

import warnings
# Workshop çıktısında gereksiz uyarıları azaltmak için kullanılır.

warnings.filterwarnings("ignore")
# Kritik hatalar görünür kalırken uzun uyarı mesajları azaltılır.

import numpy as np
# pStandard hesaplaması ve sayısal kalite kontrolleri için kullanılır.

import pandas as pd
# Aktivite, assay ve molekül tablolarını düzenlemek için kullanılır.

import requests
# ChEMBL REST API üzerinden veri indirmek için kullanılır.

from tqdm.auto import tqdm
# Assay ve molekül işlemlerinde ilerleme çubuğu göstermek için kullanılır.

from IPython.display import display
# DataFrame tablolarını Colab içinde okunabilir biçimde göstermek için kullanılır.

from google.colab import drive
# Üretilen CSV dosyalarını Google Drive'a kaydetmek için kullanılır.

from rdkit import Chem
# SMILES metinlerini RDKit molekül nesnesine dönüştürmek için kullanılır.

from rdkit.Chem.MolStandardize import rdMolStandardize
# Tuzları ve küçük karşı iyonları ayırarak parent molekül üretmek için kullanılır.

## Hücre 3 — Google Drive bağlantısı

Workshop çıktılarının kaydedileceği Google Drive alanı bağlanır.

In [ ]:
drive.mount("/content/drive", force_remount=False)
# Google Drive standart Colab dizinine bağlanır.

## Hücre 4 — Target ve API ayarları

Target ID, aktivite tipi, assay filtresi, sayfalama ve Drive çıktı klasörü tanımlanır.

In [ ]:
TARGET_ID = "CHEMBL206"
# Workshopta çalışılacak ChEMBL Target ID burada değiştirilir.

STANDARD_TYPE = "IC50"
# Regression endpointi yalnızca IC50 ölçümleriyle sınırlandırılır.

REQUIRE_SINGLE_PROTEIN_FORMAT = True
# True olduğunda yalnızca single protein format assay kayıtları tutulur.

REQUIRE_EXACT_RELATION = True
# True olduğunda yalnızca standard_relation değeri '=' olan kesin ölçümler tutulur.

PAGE_SIZE = 1000
# ChEMBL API'den her sayfada istenecek maksimum kayıt sayısı belirlenir.

REQUEST_SLEEP = 0.05
# Ardışık API istekleri arasında sunucuyu zorlamamak için kısa bekleme belirlenir.

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
# ChEMBL REST API temel adresi tanımlanır.

DRIVE_ROOT = Path("/content/drive/MyDrive/MOL_FET_regression_workshop")
# Beş notebookun bütün çıktılarının kaydedileceği ortak Google Drive klasörü tanımlanır.

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
# Ortak Drive klasörü yoksa oluşturulur.

SESSION = requests.Session()
# Tekrar kullanılabilir HTTP oturumu oluşturulur.

SESSION.headers.update({"Accept": "application/json"})
# ChEMBL servisinden JSON yanıt beklendiği belirtilir.

## Hücre 5 — Aktivite birim dönüşümleri

IC50 değerlerinin molar birime çevrilmesinde kullanılacak katsayılar tanımlanır.

In [ ]:
UNIT_TO_MOLAR = {
    "M": 1.0,
    # Molar değer değişmeden kullanılır.

    "mM": 1e-3,
    # Milimolar değer molara çevrilir.

    "uM": 1e-6,
    # ASCII mikroM gösterimi molara çevrilir.

    "µM": 1e-6,
    # Mikro işaretli birim molara çevrilir.

    "μM": 1e-6,
    # Yunan mu karakterli birim molara çevrilir.

    "nM": 1e-9,
    # Nanomolar değer molara çevrilir.

    "pM": 1e-12,
    # Pikomolar değer molara çevrilir.
}
# Desteklenen aktivite birimleri ve molar dönüşüm katsayıları tanımlanır.

## Hücre 6 — Bölüm başlığı yardımcı fonksiyonu

Notebook çıktısında okunabilir bölüm başlıkları oluşturulur.

In [ ]:
def note(title, message=""):
    """Workshop çıktısında okunabilir bölüm başlığı oluşturur."""

    print("\n" + "=" * 100)
    # Üst ayırıcı çizgi yazdırılır.

    print(title)
    # Bölüm başlığı yazdırılır.

    print("=" * 100)
    # Alt ayırıcı çizgi yazdırılır.

    if message:
        # Ek açıklama olup olmadığı kontrol edilir.

        print(message)
        # Ek açıklama yazdırılır.

## Hücre 7 — Güvenli ChEMBL API isteği

Geçici ağ hatalarında yeniden deneme yapan JSON indirme fonksiyonu tanımlanır.

In [ ]:
def get_json(url, params=None, timeout=90, max_retries=4):
    """ChEMBL API yanıtını yeniden deneme desteğiyle JSON olarak döndürür."""

    last_error = None
    # Son hata daha sonra raporlanmak üzere saklanır.

    for attempt in range(1, max_retries + 1):
        # İstek belirlenen maksimum deneme sayısı kadar tekrarlanır.

        try:
            # Ağ isteği sırasında oluşabilecek hatalar yakalanır.

            response = SESSION.get(url, params=params, timeout=timeout)
            # ChEMBL REST servisine GET isteği gönderilir.

            response.raise_for_status()
            # HTTP hata kodlarında exception üretilir.

            return response.json()
            # Başarılı yanıt Python sözlüğü olarak döndürülür.

        except Exception as error:
            # Ağ veya JSON hatası yakalanır.

            last_error = error
            # Son hata saklanır.

            if attempt == max_retries:
                # Son denemeye ulaşılıp ulaşılmadığı kontrol edilir.

                break
                # Son deneme de başarısızsa döngü sonlandırılır.

            wait_seconds = attempt * 2
            # Her yeni denemede bekleme süresi artırılır.

            print(
                f"API denemesi {attempt} başarısız: {error}. "
                f"{wait_seconds} saniye sonra yeniden denenecek."
            )
            # Geçici hata kullanıcıya bildirilir.

            time.sleep(wait_seconds)
            # Yeni denemeden önce beklenir.

    raise RuntimeError(f"ChEMBL API yanıtı alınamadı: {last_error}")
    # Bütün denemeler başarısızsa açıklayıcı hata üretilir.

## Hücre 8 — Sayfalı ChEMBL veri indirme

Bir endpointteki bütün sayfaları indirip tek listede birleştiren fonksiyon tanımlanır.

In [ ]:
def fetch_paginated(endpoint, result_key, params=None, page_size=1000):
    """ChEMBL sayfalı endpointindeki bütün kayıtları indirir."""

    query = dict(params or {})
    # Kullanıcının filtreleri yeni bir sözlüğe kopyalanır.

    query["limit"] = page_size
    # Sayfa büyüklüğü sorguya eklenir.

    query["offset"] = 0
    # İlk sayfa başlangıcı sıfır olarak belirlenir.

    url = f"{CHEMBL_BASE}/{endpoint}.json"
    # İstenen endpointin JSON adresi oluşturulur.

    all_rows = []
    # Bütün sayfalardaki kayıtları tutacak liste oluşturulur.

    while True:
        # Son sayfaya ulaşılana kadar devam eden döngü başlatılır.

        payload = get_json(url, params=query)
        # İlgili sayfanın JSON yanıtı indirilir.

        page_rows = payload.get(result_key, [])
        # Yanıttaki kayıt listesi güvenli biçimde alınır.

        all_rows.extend(page_rows)
        # Sayfadaki kayıtlar toplam listeye eklenir.

        print(f"İndirilen toplam kayıt: {len(all_rows):,}", end="\r")
        # İndirilen toplam kayıt sayısı aynı satırda güncellenir.

        next_page = payload.get("page_meta", {}).get("next")
        # Sonraki sayfa bilgisi alınır.

        if not next_page or not page_rows:
            # Sonraki sayfa yoksa veya boş sayfa geldiyse kontrol edilir.

            break
            # Sayfalama döngüsü sonlandırılır.

        query["offset"] += page_size
        # Bir sonraki sayfanın başlangıç konumu artırılır.

        time.sleep(REQUEST_SLEEP)
        # Ardışık API istekleri arasında kısa süre beklenir.

    print()
    # İlerleme yazısından sonra yeni satıra geçilir.

    return all_rows
    # Bütün kayıtlar liste olarak döndürülür.

## Hücre 9 — pStandard hesaplama

Öncelikle pChEMBL kullanılır; yoksa IC50 değeri molar birime çevrilerek pStandard hesaplanır.

In [ ]:
def calculate_pstandard(row):
    """Bir aktivite satırından pStandard/pIC50 hesaplar."""

    pchembl_value = pd.to_numeric(row.get("pchembl_value"), errors="coerce")
    # ChEMBL tarafından hesaplanmış pChEMBL değeri sayısala çevrilir.

    if pd.notna(pchembl_value):
        # Geçerli pChEMBL değeri olup olmadığı kontrol edilir.

        return float(pchembl_value)
        # Geçerli değer doğrudan pStandard olarak kullanılır.

    standard_value = pd.to_numeric(row.get("standard_value"), errors="coerce")
    # Ham standard_value sayısala çevrilir.

    standard_units = str(row.get("standard_units", "")).strip()
    # Aktivite birimi temiz metne dönüştürülür.

    if pd.isna(standard_value) or standard_units not in UNIT_TO_MOLAR:
        # Değer veya birim kullanılamıyorsa kontrol edilir.

        return np.nan
        # pStandard üretilemediği için NaN döndürülür.

    value_molar = float(standard_value) * UNIT_TO_MOLAR[standard_units]
    # IC50 değeri molar birime çevrilir.

    if value_molar <= 0:
        # Sıfır veya negatif değer kontrol edilir.

        return np.nan
        # Fiziksel olarak geçersiz değer çıkarılır.

    return float(-np.log10(value_molar))
    # pStandard = -log10(IC50[M]) formülü uygulanır.

## Hücre 10 — Parent SMILES standardizasyonu

SMILES doğrulanır, tuzlar ve küçük karşı iyonlar çıkarılır, canonical parent SMILES üretilir.

In [ ]:
def standardize_parent_smiles(smiles):
    """SMILES'ı doğrular, tuzları çıkarır ve canonical parent SMILES üretir."""

    if pd.isna(smiles):
        # Eksik SMILES kontrol edilir.

        return None
        # Eksik değer geçersiz kabul edilir.

    text = str(smiles).strip()
    # SMILES temiz metne dönüştürülür.

    if not text:
        # Boş metin kontrol edilir.

        return None
        # Boş SMILES geçersiz kabul edilir.

    try:
        # RDKit ayrıştırma hatalarını yakalamak için try bloğu başlatılır.

        molecule = Chem.MolFromSmiles(text)
        # SMILES RDKit molekülüne dönüştürülür.

        if molecule is None:
            # RDKit'in molekül üretip üretmediği kontrol edilir.

            return None
            # Ayrıştırılamayan SMILES geçersiz kabul edilir.

        parent = rdMolStandardize.FragmentParent(molecule)
        # Tuz ve küçük karşı iyonlar çıkarılarak ana fragman seçilir.

        Chem.SanitizeMol(parent)
        # Parent molekülün kimyasal geçerliliği kontrol edilir.

        return Chem.MolToSmiles(parent, canonical=True, isomericSmiles=True)
        # Parent yapı canonical ve stereokimyayı koruyan SMILES olarak döndürülür.

    except Exception:
        # Herhangi bir RDKit standardizasyon hatası yakalanır.

        return None
        # Hatalı yapı geçersiz kabul edilir.

## Hücre 11 — Güvenli sütun okuma

Bir sütun mevcutsa kopyasını, yoksa aynı indeksli boş bir Seri döndüren fonksiyon tanımlanır.

In [ ]:
def get_series(dataframe, column_name):
    """Sütun varsa döndürür, yoksa aynı indeksli boş Seri oluşturur."""

    if column_name in dataframe.columns:
        # İstenen sütunun veri setinde olup olmadığı kontrol edilir.

        return dataframe[column_name].copy()
        # Mevcut sütunun kopyası döndürülür.

    return pd.Series(index=dataframe.index, dtype="object")
    # Sütun yoksa aynı indeksli boş Seri döndürülür.

## Hücre 12 — Target metadata doğrulaması

Girilen ChEMBL Target ID doğrulanır ve target bilgileri tablo halinde gösterilir.

In [ ]:
note("1. TARGET METADATA", "Girilen ChEMBL Target ID doğrulanıyor.")
# Target metadata bölümü başlatılır.

target_json = get_json(f"{CHEMBL_BASE}/target/{TARGET_ID}.json")
# Target metadata ChEMBL endpointinden indirilir.

target_metadata = pd.DataFrame([
    {
        "target_chembl_id": target_json.get("target_chembl_id"),
        # Target ChEMBL kimliği saklanır.

        "pref_name": target_json.get("pref_name"),
        # Target tercih edilen adı saklanır.

        "organism": target_json.get("organism"),
        # Target organizması saklanır.

        "target_type": target_json.get("target_type"),
        # Target tipi saklanır.
    }
])
# Target metadata tek satırlık DataFrame'e dönüştürülür.

display(target_metadata)
# Target metadata ekranda gösterilir.

if target_metadata.loc[0, "target_chembl_id"] != TARGET_ID:
    # API yanıtındaki Target ID kullanıcı girdisiyle karşılaştırılır.

    raise ValueError("Girilen TARGET_ID ChEMBL tarafından doğrulanamadı.")
    # Target eşleşmiyorsa pipeline durdurulur.

print("✅ CHECKPOINT 00.2: Target metadata doğrulandı.")
# Target metadata adımının tamamlandığı bildirilir.

## Hücre 13 — IC50 aktivitelerinin indirilmesi

Target için bütün IC50 aktiviteleri ChEMBL API üzerinden sayfalı olarak indirilir.

In [ ]:
note("2. IC50 AKTİVİTELERİ", "Target için bütün IC50 kayıtları indiriliyor.")
# Aktivite indirme bölümü başlatılır.

activity_rows = fetch_paginated(
    endpoint="activity",
    result_key="activities",
    params={
        "target_chembl_id": TARGET_ID,
        # Yalnızca seçilen targeta ait aktiviteler istenir.

        "standard_type": STANDARD_TYPE,
        # Yalnızca IC50 standard_type kayıtları istenir.
    },
    page_size=PAGE_SIZE,
)
# ChEMBL aktivite endpointinden filtrelenmiş kayıtlar indirilir.

df_activity_raw = pd.DataFrame(activity_rows)
# Aktivite listesi DataFrame'e dönüştürülür.

if df_activity_raw.empty:
    # Aktivite tablosunun boş olup olmadığı kontrol edilir.

    raise ValueError(f"{TARGET_ID} için IC50 aktivitesi bulunamadı.")
    # Hiç aktivite yoksa pipeline durdurulur.

if "activity_id" in df_activity_raw.columns:
    # activity_id sütununun bulunup bulunmadığı kontrol edilir.

    df_activity_raw = df_activity_raw.drop_duplicates(
        subset=["activity_id"], keep="first"
    ).copy()
    # Aynı activity_id ile tekrar eden kayıtlar çıkarılır.

print("Ham aktivite boyutu:", df_activity_raw.shape)
# Ham aktivite tablosunun boyutu yazdırılır.

display(df_activity_raw.head())
# İlk kayıtlar ekranda gösterilir.

print("✅ CHECKPOINT 00.3: Ham IC50 aktiviteleri indirildi.")
# Aktivite indirme adımının tamamlandığı bildirilir.

## Hücre 14 — Assay metadata fonksiyonu

Aynı assay için tekrar istek göndermemek üzere cache destekli assay metadata fonksiyonu tanımlanır.

In [ ]:
note("3. SINGLE PROTEIN FORMAT", "Assay BAO formatları kontrol ediliyor.")
# Assay format kontrol bölümü başlatılır.

assay_cache = {}
# Aynı assay için tekrar API isteği göndermemek üzere cache oluşturulur.


def fetch_assay_metadata(assay_id):
    """Assay BAO format bilgisini cache desteğiyle indirir."""

    if assay_id in assay_cache:
        # Assay daha önce sorgulandıysa kontrol edilir.

        return assay_cache[assay_id]
        # Önceden indirilen metadata döndürülür.

    assay_json = get_json(f"{CHEMBL_BASE}/assay/{assay_id}.json")
    # Assay metadata endpointinden indirilir.

    result = {
        "assay_chembl_id": assay_id,
        # Assay kimliği saklanır.

        "bao_label_assay": assay_json.get("bao_label"),
        # BAO etiket bilgisi saklanır.

        "bao_format_assay": assay_json.get("bao_format"),
        # BAO format kimliği saklanır.

        "assay_type_assay": assay_json.get("assay_type"),
        # Assay tipi kalite kontrol amacıyla saklanır.
    }
    # Assay metadata sözlüğü oluşturulur.

    assay_cache[assay_id] = result
    # Sonuç cache içine eklenir.

    return result
    # Assay metadata döndürülür.

## Hücre 15 — Assay metadata ve single protein format filtresi

Benzersiz assay kimliklerinin BAO bilgileri indirilir, aktivite tablosuna eklenir ve single protein format kayıtları seçilir.

In [ ]:
if REQUIRE_SINGLE_PROTEIN_FORMAT:
    # Single protein format filtresinin açık olup olmadığı kontrol edilir.

    unique_assays = sorted(
        df_activity_raw["assay_chembl_id"].dropna().astype(str).unique()
    )
    # Aktivite tablosundaki benzersiz assay kimlikleri seçilir.

    assay_rows = []
    # Assay metadata kayıtlarını tutacak liste oluşturulur.

    for assay_id in tqdm(unique_assays, desc="Assay formatları"):
        # Bütün benzersiz assayler ilerleme çubuğuyla işlenir.

        assay_rows.append(fetch_assay_metadata(assay_id))
        # Assay metadata listeye eklenir.

        time.sleep(REQUEST_SLEEP)
        # Ardışık API istekleri arasında kısa süre beklenir.

    df_assay = pd.DataFrame(assay_rows)
    # Assay metadata listesi DataFrame'e dönüştürülür.

    df_activity = df_activity_raw.merge(
        df_assay, on="assay_chembl_id", how="left"
    )
    # Assay metadata aktivite tablosuna eklenir.

    bao_label = get_series(df_activity, "bao_label").fillna(
        get_series(df_activity, "bao_label_assay")
    )
    # Aktivite BAO etiketi eksikse assay endpointinden tamamlanır.

    bao_format = get_series(df_activity, "bao_format").fillna(
        get_series(df_activity, "bao_format_assay")
    )
    # Aktivite BAO formatı eksikse assay endpointinden tamamlanır.

    single_mask = (
        bao_label.astype(str).str.strip().str.lower().eq("single protein format")
        | bao_format.astype(str).str.contains("BAO_0000357", regex=False, na=False)
    )
    # Single protein format etiketi veya BAO kimliği taşıyan kayıtlar belirlenir.

    df_activity = df_activity.loc[single_mask].copy()
    # Yalnızca single protein format aktiviteleri tutulur.

else:
    # Single protein filtresi kapalıysa alternatif kola geçilir.

    df_activity = df_activity_raw.copy()
    # Bütün IC50 aktiviteleri tutulur.

if df_activity.empty:
    # Assay filtresinden sonra kayıt kalıp kalmadığı kontrol edilir.

    raise ValueError("Single protein format filtresinden sonra aktivite kalmadı.")
    # Kayıt kalmadıysa pipeline durdurulur.

print("Single protein format sonrası kayıt:", len(df_activity))
# Assay filtresi sonrası kayıt sayısı yazdırılır.

print("✅ CHECKPOINT 00.4: Single protein format filtresi tamamlandı.")
# Assay filtresi adımının tamamlandığı bildirilir.

## Hücre 16 — Aktivite kalite filtresi

Kesin `=` ilişkili kayıtlar seçilir, pStandard üretilir ve eksik/geçersiz aktiviteler çıkarılır.

In [ ]:
note("4. AKTİVİTE KALİTESİ", "Kesin ilişki ve pStandard kontrolleri uygulanıyor.")
# Aktivite kalite kontrol bölümü başlatılır.

if REQUIRE_EXACT_RELATION:
    # Yalnızca kesin eşitlik ölçümleri isteniyor mu kontrol edilir.

    exact_mask = df_activity["standard_relation"].astype(str).str.strip().eq("=")
    # standard_relation değeri '=' olan satırlar belirlenir.

    df_activity = df_activity.loc[exact_mask].copy()
    # Sansürlü <, > ve benzeri ölçümler çıkarılır.


df_activity["pStandard"] = df_activity.apply(calculate_pstandard, axis=1)
# Her aktivite satırı için pStandard hesaplanır.

df_activity = df_activity.dropna(
    subset=["molecule_chembl_id", "assay_chembl_id", "pStandard"]
).copy()
# Molekül ID, assay ID veya pStandard eksik olan satırlar çıkarılır.

df_activity = df_activity[np.isfinite(df_activity["pStandard"])].copy()
# Sonsuz pStandard değerleri çıkarılır.

if df_activity.empty:
    # Kalite filtresinden sonra kayıt kalıp kalmadığı kontrol edilir.

    raise ValueError("Aktivite kalite filtresinden sonra kayıt kalmadı.")
    # Kayıt kalmadıysa pipeline durdurulur.

print("Kalite filtresi sonrası kayıt:", len(df_activity))
# Kalite filtresi sonrası kayıt sayısı yazdırılır.

print(
    "pStandard aralığı:",
    float(df_activity["pStandard"].min()),
    "—",
    float(df_activity["pStandard"].max()),
)
# pStandard aralığı yazdırılır.

print("✅ CHECKPOINT 00.5: Aktivite kalite filtresi tamamlandı.")
# Aktivite kalite adımının tamamlandığı bildirilir.

## Hücre 17 — Ligand yapı alanlarının hazırlanması

Aktivite yanıtındaki canonical SMILES ve parent molekül ID alanları güvenli biçimde alınır.

In [ ]:
note("5. LİGAND SMILES", "SMILES ve parent molekül bilgileri dikkatli biçimde hazırlanıyor.")
# Ligand SMILES hazırlama bölümü başlatılır.

canonical_activity_smiles = get_series(df_activity, "canonical_smiles")
# Aktivite yanıtındaki canonical_smiles sütunu güvenli biçimde alınır.

parent_activity_ids = get_series(df_activity, "parent_molecule_chembl_id")
# Aktivite yanıtındaki parent molekül ID sütunu güvenli biçimde alınır.


df_activity["source_smiles"] = canonical_activity_smiles
# Aktivite içindeki SMILES başlangıç kaynak SMILES olarak atanır.

df_activity["parent_chembl_id"] = parent_activity_ids
# Aktivite içindeki parent ID başlangıç parent kimliği olarak atanır.

missing_structure_mask = (
    df_activity["source_smiles"].isna()
    | df_activity["source_smiles"].astype(str).str.strip().eq("")
    | df_activity["parent_chembl_id"].isna()
)
# SMILES veya parent ID bilgisi eksik olan satırlar belirlenir.

molecule_cache = {}
# Aynı molekül için tekrar API isteği göndermemek üzere cache oluşturulur.

## Hücre 18 — Eksik ligand bilgisi için fallback fonksiyonu

Eksik SMILES veya parent ID bilgilerini ChEMBL molekül endpointinden tamamlayan cache destekli fonksiyon tanımlanır.

In [ ]:
def fetch_molecule_fallback(molecule_id):
    """Eksik SMILES ve parent ID bilgisini molekül endpointinden tamamlar."""

    if molecule_id in molecule_cache:
        # Molekül daha önce sorgulandıysa kontrol edilir.

        return molecule_cache[molecule_id]
        # Önceden indirilen sonuç döndürülür.

    molecule_json = get_json(f"{CHEMBL_BASE}/molecule/{molecule_id}.json")
    # Molekül metadata endpointinden indirilir.

    hierarchy = molecule_json.get("molecule_hierarchy") or {}
    # Molekül hiyerarşisi güvenli biçimde alınır.

    parent_id = hierarchy.get("parent_chembl_id") or molecule_id
    # Parent ID varsa kullanılır, yoksa kaynak molekül ID kullanılır.

    structures = molecule_json.get("molecule_structures") or {}
    # Kaynak molekül yapısı güvenli biçimde alınır.

    source_smiles = structures.get("canonical_smiles")
    # Kaynak molekül canonical SMILES değeri alınır.

    result = {
        "molecule_chembl_id": molecule_id,
        # Kaynak molekül ID saklanır.

        "parent_chembl_id_fallback": parent_id,
        # Parent molekül ID saklanır.

        "canonical_smiles_fallback": source_smiles,
        # Kaynak canonical SMILES saklanır.
    }
    # Fallback sonuç sözlüğü oluşturulur.

    molecule_cache[molecule_id] = result
    # Sonuç cache içine eklenir.

    return result
    # Fallback sonuç döndürülür.

## Hücre 19 — Eksik ligand yapılarının tamamlanması

Eksik molekül kayıtları API üzerinden indirilir ve aktivite tablosuna eklenir.

In [ ]:
if missing_structure_mask.any():
    # Eksik yapı bilgisine sahip kayıt olup olmadığı kontrol edilir.

    missing_molecule_ids = sorted(
        df_activity.loc[missing_structure_mask, "molecule_chembl_id"]
        .dropna()
        .astype(str)
        .unique()
    )
    # Eksik yapı bilgisine sahip benzersiz molekül kimlikleri seçilir.

    fallback_rows = []
    # Fallback sonuçlarını tutacak liste oluşturulur.

    for molecule_id in tqdm(missing_molecule_ids, desc="Eksik ligand yapıları"):
        # Eksik moleküller ilerleme çubuğuyla işlenir.

        fallback_rows.append(fetch_molecule_fallback(molecule_id))
        # Molekül fallback sonucu listeye eklenir.

        time.sleep(REQUEST_SLEEP)
        # Ardışık API istekleri arasında kısa süre beklenir.

    if fallback_rows:
        # En az bir fallback kaydı varsa kontrol edilir.

        df_fallback = pd.DataFrame(fallback_rows)
        # Fallback listesi DataFrame'e dönüştürülür.

        df_activity = df_activity.merge(
            df_fallback, on="molecule_chembl_id", how="left"
        )
        # Fallback bilgileri aktivite tablosuna eklenir.

        df_activity["source_smiles"] = df_activity["source_smiles"].fillna(
            df_activity["canonical_smiles_fallback"]
        )
        # Eksik SMILES değerleri molekül endpointinden tamamlanır.

        df_activity["parent_chembl_id"] = df_activity["parent_chembl_id"].fillna(
            df_activity["parent_chembl_id_fallback"]
        )
        # Eksik parent ID değerleri molekül endpointinden tamamlanır.

## Hücre 20 — Parent ID ve parent SMILES üretimi

Eksik parent ID değerleri tamamlanır, SMILES standardize edilir ve geçersiz ligandlar çıkarılır.

In [ ]:
df_activity["parent_chembl_id"] = df_activity["parent_chembl_id"].fillna(
    df_activity["molecule_chembl_id"]
)
# Hâlâ eksik parent ID varsa kaynak molekül ID ile tamamlanır.


df_activity["parent_smiles"] = df_activity["source_smiles"].apply(
    standardize_parent_smiles
)
# Her ligand SMILES tuzlardan arındırılmış canonical parent SMILES'a dönüştürülür.


df_activity = df_activity.dropna(
    subset=["parent_chembl_id", "parent_smiles"]
).copy()
# Parent ID veya geçerli parent SMILES üretilemeyen satırlar çıkarılır.

if df_activity.empty:
    # SMILES standardizasyonundan sonra kayıt kalıp kalmadığı kontrol edilir.

    raise ValueError("SMILES doğrulamasından sonra kullanılabilir ligand kalmadı.")
    # Ligand kalmadıysa pipeline durdurulur.

print("Geçerli benzersiz parent SMILES:", df_activity["parent_smiles"].nunique())
# Geçerli benzersiz parent yapı sayısı yazdırılır.

print("✅ CHECKPOINT 00.6: Ligand SMILES ve parent yapıları hazırlandı.")
# Ligand yapı adımının tamamlandığı bildirilir.

## Hücre 21 — Parent yapı bazında aggregation

Aynı parent ID ve parent SMILES'a ait ölçümler medyan, ortalama ve standart sapma ile özetlenir.

In [ ]:
note("6. PARENT DEDUP", "Aynı parent yapıya ait ölçümler medyanla birleştiriliyor.")
# Parent-dedup bölümü başlatılır.


df_clean = (
    df_activity.groupby(["parent_chembl_id", "parent_smiles"], as_index=False)
    .agg(
        pStandard=("pStandard", "median"),
        # Model targetı için robust medyan pStandard kullanılır.

        pStandard_mean=("pStandard", "mean"),
        # Karşılaştırma amacıyla ortalama pStandard saklanır.

        pStandard_std=("pStandard", "std"),
        # Ölçümler arası standart sapma saklanır.

        pStandard_min=("pStandard", "min"),
        # En düşük pStandard değeri saklanır.

        pStandard_max=("pStandard", "max"),
        # En yüksek pStandard değeri saklanır.

        n_measurements=("pStandard", "size"),
        # Parent yapı için toplam aktivite ölçümü sayısı saklanır.

        n_assays=("assay_chembl_id", "nunique"),
        # Parent yapı için benzersiz assay sayısı saklanır.

        n_molecule_ids=("molecule_chembl_id", "nunique"),
        # Parent yapı altında birleşen benzersiz molekül ID sayısı saklanır.
    )
)
# Aktivite kayıtları parent ID ve parent SMILES bazında özetlenir.

## Hücre 22 — Aktivite dağılımı ve duplicate kontrolü

pStandard aralığı hesaplanır, deneysel çatışmalar işaretlenir ve tekrar kontrolü yapılır.

In [ ]:
df_clean["pStandard_std"] = df_clean["pStandard_std"].fillna(0.0)
# Tek ölçümlü yapılardaki NaN standart sapma sıfırla doldurulur.


df_clean["pStandard_range"] = (
    df_clean["pStandard_max"] - df_clean["pStandard_min"]
)
# Aynı parent yapıdaki ölçümler arası aktivite aralığı hesaplanır.


df_clean["activity_conflict_flag"] = df_clean["pStandard_range"] > 1.0
# Bir log birimden büyük deneysel saçılım gösteren yapılar işaretlenir.


df_clean = df_clean.sort_values(
    ["parent_chembl_id", "parent_smiles"]
).reset_index(drop=True)
# Final veri seti tekrar üretilebilir sıraya göre düzenlenir.


if df_clean.duplicated(subset=["parent_chembl_id", "parent_smiles"]).any():
    # Parent ID ve SMILES ikilisinde tekrar kalıp kalmadığı kontrol edilir.

    raise AssertionError("Parent-dedup sonrasında tekrar eden kayıt bulundu.")
    # Tekrar bulunursa pipeline durdurulur.

## Hücre 23 — Sade target–SMILES tablosu

Workshop için Target ID, parent ChEMBL ID, pStandard ve parent SMILES içeren sade tablo hazırlanır.

In [ ]:
df_target_smiles = df_clean[
    ["parent_chembl_id", "pStandard", "parent_smiles"]
].copy()
# Workshop için sade target-SMILES tablosu oluşturulur.


df_target_smiles.insert(0, "target_chembl_id", TARGET_ID)
# Sade tablonun ilk sütununa Target ID eklenir.


print("Final parent-dedup boyutu:", df_clean.shape)
# Final veri setinin boyutu yazdırılır.


display(df_clean.head(10))
# Final tablonun ilk 10 satırı gösterilir.

print("✅ CHECKPOINT 00.7: Parent-dedup regression veri seti hazırlandı.")
# Parent-dedup adımının tamamlandığı bildirilir.

## Hücre 24 — Google Drive çıktı yolları

Ana regression CSV, sade target–SMILES CSV ve target metadata CSV dosya yolları oluşturulur.

In [ ]:
note("7. GOOGLE DRIVE KAYDI", "Üretilen dosyalar ortak Drive klasörüne kaydediliyor.")
# Google Drive kayıt bölümü başlatılır.


main_filename = f"{TARGET_ID}_IC50_single_protein_format_CLEAN_parent_dedup.csv"
# GitHub data klasörüne yüklenecek ana CSV adı oluşturulur.


target_smiles_filename = f"{TARGET_ID}_IC50_target_smiles.csv"
# Sade target-SMILES CSV adı oluşturulur.


metadata_filename = f"{TARGET_ID}_target_metadata.csv"
# Target metadata CSV adı oluşturulur.


main_path = DRIVE_ROOT / main_filename
# Ana regression CSV için Drive yolu oluşturulur.


target_smiles_path = DRIVE_ROOT / target_smiles_filename
# Sade target-SMILES CSV için Drive yolu oluşturulur.


metadata_path = DRIVE_ROOT / metadata_filename
# Target metadata CSV için Drive yolu oluşturulur.

## Hücre 25 — CSV dosyalarının Drive'a kaydedilmesi

Üç çıktı Google Drive'a kaydedilir, dosya varlığı ve boyutu doğrulanır.

In [ ]:
df_clean.to_csv(main_path, index=False)
# Ana regression veri seti Google Drive'a kaydedilir.


df_target_smiles.to_csv(target_smiles_path, index=False)
# Sade target-SMILES tablosu Google Drive'a kaydedilir.


target_metadata.to_csv(metadata_path, index=False)
# Target metadata tablosu Google Drive'a kaydedilir.


for output_path in [main_path, target_smiles_path, metadata_path]:
    # Üretilen her dosya sırayla kontrol edilir.

    if not output_path.exists() or output_path.stat().st_size == 0:
        # Dosyanın oluşup oluşmadığı ve boş olup olmadığı kontrol edilir.

        raise IOError(f"Drive çıktısı oluşturulamadı: {output_path}")
        # Hatalı çıktı varsa pipeline durdurulur.

    print("[Drive kaydı]", output_path)
    # Başarılı Drive kayıt yolu yazdırılır.


print("\nGitHub data/ klasörüne yükleyiniz:", main_filename)
# Sonraki notebookun GitHub'dan okuyacağı ana dosya bildirilir.

print("✅ CHECKPOINT 00.8: Notebook 00 tamamlandı ve bütün çıktılar Drive'a kaydedildi.")
# Notebookun başarıyla tamamlandığı bildirilir.

## Notebook sonu

Başarılı çalıştırma sonunda aşağıdaki dosyalar Google Drive içindeki ortak workshop klasörüne kaydedilir:

```text
{TARGET_ID}_IC50_single_protein_format_CLEAN_parent_dedup.csv
{TARGET_ID}_IC50_target_smiles.csv
{TARGET_ID}_target_metadata.csv
```

Bir sonraki notebookun GitHub'dan okuyacağı ana dosya:

```text
{TARGET_ID}_IC50_single_protein_format_CLEAN_parent_dedup.csv
```